# Lesson 14.2: Separate capability discovery from execution checks

Discovery is finding out which operations a service offers. Execution is asking it to perform a particular operation with particular arguments. In the workshop, follow a permitted read and a refused handoff. Discovering a name does not confer the right to use it.

[Companion lesson](../modules/14-agent-integration/02-code-workshop.md) · [Original-source walkthrough](../modules/14-agent-integration/05-source-walkthrough.md)

Run one cell at a time from top to bottom. The programme uses the fictional records shown in the code. No API key or model download is needed.

## Predict the result

Name the capability the example discovers and the capability it refuses. What can this caller read?

In [ ]:
CATALOGUE = {
    "read_case": {"required_scope": "case:read"},
    "prepare_handoff": {"required_scope": "case:draft"},
}
CASES = {"C-17": {"order_id": "O-17", "status": "disputed"}}

def discover(allowed_names):
    return {name: CATALOGUE[name] for name in allowed_names if name in CATALOGUE}

def call_tool(name, arguments, principal):
    if name not in CATALOGUE:
        raise ValueError("Unknown capability")
    required = CATALOGUE[name]["required_scope"]
    if required not in principal["scopes"]:
        raise PermissionError("Operation is not authorised")
    if set(arguments) != {"case_id"}:
        raise ValueError("Unexpected argument fields")
    case_id = arguments["case_id"]
    if not isinstance(case_id, str) or case_id not in principal["case_ids"]:
        raise PermissionError("Case is outside the permitted scope")
    if case_id not in CASES:
        raise LookupError("Case not found")
    if name == "read_case":
        return {"case_id": case_id, "record": dict(CASES[case_id]), "source": "cases-v1"}
    return {"case_id": case_id, "status": "simulated_draft", "source": "cases-v1"}

reader = {"scopes": {"case:read"}, "case_ids": {"C-17"}}
print(list(discover({"read_case"})))
print(call_tool("read_case", {"case_id": "C-17"}, reader)["record"]["status"])
try:
    call_tool("prepare_handoff", {"case_id": "C-17"}, reader)
except PermissionError as error:
    print(str(error))

## Inspect the result

`print` displays values below the cell. When present, `assert` checks a specific expectation and raises an error if it fails.

In [ ]:
assert list(discover({"read_case"})) == ["read_case"]

## Try a change

Keep the same caller, but request another case through the permitted read tool. Should the service return it?

Make a prediction before opening the worked answer.

In [ ]:
# Try the change here, or make a temporary edit in the first code cell.

<details>
<summary>Worked answer</summary>

Permission to call a read tool does not give the caller access to every case.

```python
try:
    call_tool("read_case", {"case_id": "C-18"}, reader)
except PermissionError as error:
    print(error)
else:
    raise AssertionError("Reading the other case should be denied")
```

</details>